# Safe Driver Prediction: Systematic Evaluation of Deep Learning and Classical Models

> Objective: Conduct a rigorous and reproducible comparison of TabNet, FT-Transformer, and classical models (XGBoost, LightGBM, CatBoost, Logistic Regression, Random Forest).

## Experimental Protocol (Strictly Enforced)
- Fixed data split: Train/Valid/Test = 60%/20%/20%, and all methods must share the same split
- Random seeds: at least 3 seeds, report mean and standard deviation
- Hyperparameter budget: 10 Optuna trials per method (configurable via global variables)
- Classification metrics: Accuracy, AUC-ROC, F1-score
- Practical evaluation: preprocessing complexity, training cost, inference latency, model size, stability, interpretability

## Notebook Stages
1. Data cleaning and preprocessing
2. Feature engineering
3. Model training and tuning (separate block per model)
4. Prediction and comprehensive analysis

In [ ]:
# ===== 0) Environment Setup and Global Configuration =====
import os
import gc
import time
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.INFO)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# Reproducible experiment configuration
FAST_MODE = False

if FAST_MODE:
    # Fast mode: used for quick pipeline validation with much shorter training time
    SEEDS = [123]
    N_TRIALS = 1
    MAX_TRAIN_ROWS = 80000
    MAX_VALID_ROWS = 30000
    MAX_TRAIN_ROWS_FT = 15000
    MAX_VALID_ROWS_FT = 6000
    TABNET_MAX_EPOCHS = 40
    FT_MAX_EPOCHS = 10
    INFER_REPEAT = 1
    FT_FORCE_CPU = True
else:
    # Production mode: used for final report
    SEEDS = [123, 456, 789]
    N_TRIALS = 5
    MAX_TRAIN_ROWS = None
    MAX_VALID_ROWS = None
    # Explicitly set subsampling for FT-Transformer to avoid OOM, even in non-FAST_MODE
    MAX_TRAIN_ROWS_FT = 15000 # Use a reasonable cap for FT-Transformer
    MAX_VALID_ROWS_FT = 6000  # Use a reasonable cap for FT-Transformer
    TABNET_MAX_EPOCHS = 80
    FT_MAX_EPOCHS = 20
    INFER_REPEAT = 5
    FT_FORCE_CPU = False

VALID_SIZE = 0.20  # 20% of the full dataset
TEST_SIZE = 0.20   # 20% of the full dataset
TARGET_COL = "target"

# Use  suffix consistently to avoid conflicts with FAST_MODE artifacts
DATA_PATH = Path("./dataset.csv")
RESULT_DIR = Path("./artifacts")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_INDEX_PATH = RESULT_DIR / "fixed_split_indices.json"
SUMMARY_PATH = RESULT_DIR / "model_summary.csv"
PRED_PATH = RESULT_DIR / "test_predictions_all_models.csv"
CATBOOST_TRAIN_DIR = Path("./catboost_info")
CATBOOST_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Config loaded")
print(f"[INFO] FAST_MODE: {FAST_MODE}")
print(f"[INFO] Seeds: {SEEDS}")
print(f"[INFO] Optuna trials per method: {N_TRIALS}")
print(f"[INFO] Train/Valid cap for tuning: {MAX_TRAIN_ROWS}/{MAX_VALID_ROWS}")
print(f"[INFO] FT Train/Valid cap: {MAX_TRAIN_ROWS_FT}/{MAX_VALID_ROWS_FT}")
print(f"[INFO] FT_FORCE_CPU: {FT_FORCE_CPU}")
print(f"[INFO] Result dir: {RESULT_DIR}")
print(f"[INFO] CatBoost train dir: {CATBOOST_TRAIN_DIR}")

[INFO] Config loaded
[INFO] FAST_MODE: False
[INFO] Seeds: [123, 456, 789]
[INFO] Optuna trials per method: 5
[INFO] Train/Valid cap for tuning: None/None
[INFO] FT Train/Valid cap: 15000/6000
[INFO] FT_FORCE_CPU: False
[INFO] Result dir: artifacts
[INFO] CatBoost train dir: catboost_info


In [ ]:
# Set maximum display rows
pd.set_option('display.max_rows', 500)
# Set maximum display columns
pd.set_option('display.max_columns', None)
# Set column width to avoid truncation
pd.set_option('display.max_colwidth', None)
# Set floating-point display precision
pd.set_option('display.precision', 2)

In [ ]:
import importlib
import subprocess
import sys
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

USE_GPU = torch.cuda.is_available()
print("[INFO] Core model libraries imported successfully")
print(f"[INFO] CUDA available: {USE_GPU}")
if USE_GPU:
    print(f"[INFO] CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("[WARN] CUDA is not available. GPU-specific configs will fallback to CPU.")

[INFO] Core model libraries imported successfully
[INFO] CUDA available: True
[INFO] CUDA device: Tesla T4


## 1) Data Cleaning and Preprocessing

Objectives in this stage:
- Correctly handle missing value markers (`-1`)
- Automatically identify feature types (`*_bin`, `*_cat`, and remaining numeric/ordinal features)
- Fix and save train/valid/test split indices so all models use exactly the same split
- Output data quality and missingness logs

In [ ]:
# ===== 1.1 Load Data and Basic Audit =====
assert DATA_PATH.exists(), f"dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(f"[INFO] Loaded data shape: {df.shape}")
print(f"[INFO] Columns: {len(df.columns)}")

assert TARGET_COL in df.columns, f"target column '{TARGET_COL}' not found"
print("[INFO] target distribution:")
print(df[TARGET_COL].value_counts(normalize=True).rename("ratio"))

# Treat -1 as missing values (per assignment specification)
feature_cols = [c for c in df.columns if c != TARGET_COL]
df[feature_cols] = df[feature_cols].replace(-1, np.nan)

missing_stats = df[feature_cols].isna().mean().sort_values(ascending=False)
print("[INFO] Top-20 missing ratio columns:")
print(missing_stats.head(20))

print("[INFO] Sample rows:")
display(df.head(5))

[INFO] Loaded data shape: (595212, 59)
[INFO] Columns: 59
[INFO] target distribution:
target
0    0.96
1    0.04
Name: ratio, dtype: float64
[INFO] Top-20 missing ratio columns:
ps_car_03_cat    6.91e-01
ps_car_05_cat    4.48e-01
ps_reg_03        1.81e-01
ps_car_14        7.16e-02
ps_car_07_cat    1.93e-02
ps_ind_05_cat    9.76e-03
ps_car_09_cat    9.56e-04
ps_ind_02_cat    3.63e-04
ps_car_01_cat    1.80e-04
ps_ind_04_cat    1.39e-04
ps_car_02_cat    8.40e-06
ps_car_11        8.40e-06
ps_car_12        1.68e-06
ps_ind_08_bin    0.00e+00
ps_ind_09_bin    0.00e+00
ps_ind_11_bin    0.00e+00
ps_ind_12_bin    0.00e+00
ps_ind_13_bin    0.00e+00
ps_ind_17_bin    0.00e+00
ps_ind_16_bin    0.00e+00
dtype: float64
[INFO] Sample rows:


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2.0,5,1.0,0.0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.72,10.0,1.0,NaN,0,1.0,4,1.0,0,0.0,1,12,2.0,0.40,0.88,0.37,3.61,0.6,0.5,0.2,3,1,10,1,10,1,5,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1.0,7,0.0,0.0,0,0,1,0,0,0,0,0,0,3,0,0,1,0.8,0.4,0.77,11.0,1.0,NaN,0,NaN,11,1.0,1,2.0,1,19,3.0,0.32,0.62,0.39,2.45,0.3,0.1,0.3,2,1,9,5,8,1,7,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4.0,9,1.0,0.0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,NaN,7.0,1.0,NaN,0,NaN,14,1.0,1,2.0,1,60,1.0,0.32,0.64,0.35,3.32,0.5,0.7,0.1,2,2,9,1,8,2,7,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1.0,2,0.0,0.0,1,0,0,0,0,0,0,0,0,8,1,0,0,0.9,0.2,0.58,7.0,1.0,0.0,0,1.0,11,1.0,1,3.0,1,104,1.0,0.37,0.54,0.29,2.00,0.6,0.9,0.1,2,4,7,1,8,4,2,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2.0,0,1.0,0.0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.84,11.0,1.0,NaN,0,NaN,14,1.0,1,2.0,1,82,3.0,0.32,0.57,0.37,2.00,0.4,0.6,0.0,2,2,6,3,10,2,12,3,1,1,3,0,0,0,1,1,0


In [ ]:
# ===== 1.2 Automatically Identify Feature Types =====
all_features = [c for c in df.columns if c != TARGET_COL]

bin_cols = [c for c in all_features if c.endswith("_bin")]
cat_cols = [c for c in all_features if c.endswith("_cat")]
num_cols = [c for c in all_features if c not in set(bin_cols + cat_cols)]

# Count prefix groups based on assignment hint (ind/reg/car/calc)
def get_prefix(col):
    parts = col.split("_")
    return parts[1] if len(parts) > 1 else "unknown"

prefix_group = pd.Series([get_prefix(c) for c in all_features]).value_counts()

print(f"[INFO] Binary columns: {len(bin_cols)}")
print(f"[INFO] Categorical columns: {len(cat_cols)}")
print(f"[INFO] Numeric/Ordinal columns: {len(num_cols)}")
print("[INFO] Prefix group counts:")
print(prefix_group)

# Check high-cardinality categorical features (for preprocessing complexity analysis)
high_cardinality = []
for c in cat_cols:
    n_unique = df[c].nunique(dropna=True)
    if n_unique >= 20:
        high_cardinality.append((c, n_unique))
high_cardinality = sorted(high_cardinality, key=lambda x: x[1], reverse=True)

print(f"[INFO] High-cardinality categorical features (>=20 unique): {len(high_cardinality)}")
print(high_cardinality[:20])

[INFO] Binary columns: 17
[INFO] Categorical columns: 14
[INFO] Numeric/Ordinal columns: 27
[INFO] Prefix group counts:
calc       20
ind        18
car        16
reg         3
unknown     1
Name: count, dtype: int64
[INFO] High-cardinality categorical features (>=20 unique): 1
[('ps_car_11_cat', 104)]


In [ ]:
# ===== 1.3 Fixed 60/20/20 Split (Stratified) =====
X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].astype(int).copy()

# Step 1: split out test=20%
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEEDS[0], stratify=y
)

# Step 2: split valid=20% of total from the remaining 80%
valid_ratio_in_train_valid = VALID_SIZE / (1 - TEST_SIZE)  # 0.2/0.8 = 0.25
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=valid_ratio_in_train_valid,
    random_state=SEEDS[0],
    stratify=y_train_valid
)

print("[INFO] Split summary:")
print(f"Train: {X_train.shape}, ratio={len(X_train)/len(X):.3f}")
print(f"Valid: {X_valid.shape}, ratio={len(X_valid)/len(X):.3f}")
print(f"Test : {X_test.shape}, ratio={len(X_test)/len(X):.3f}")

# Save indices for strict reproducibility
split_idx = {
    "train_index": X_train.index.tolist(),
    "valid_index": X_valid.index.tolist(),
    "test_index": X_test.index.tolist(),
}
with open(SPLIT_INDEX_PATH, "w", encoding="utf-8") as f:
    json.dump(split_idx, f, ensure_ascii=False, indent=2)
print(f"[INFO] Saved split indices to {SPLIT_INDEX_PATH}")

[INFO] Split summary:
Train: (357126, 58), ratio=0.600
Valid: (119043, 58), ratio=0.200
Test : (119043, 58), ratio=0.200
[INFO] Saved split indices to artifacts/fixed_split_indices.json


## 2) Feature Engineering and Shared Utilities

Objectives in this stage:
- Build reusable preprocessing pipelines for different model families
- Provide shared utilities for evaluation, latency benchmarking, model size estimation, and result aggregation
- Reduce duplicated code in later model-specific training blocks

In [ ]:
# ===== 2.1 Shared Preprocessors and Evaluation Utilities =====
def build_sklearn_preprocessor(model_family: str):
    """
    model_family: 'linear' | 'tree'
    linear: numeric standardization + categorical one-hot
    tree  : numeric median imputation + categorical ordinal encoding
    """
    if model_family == "linear":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ])
    elif model_family == "tree":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
    else:
        raise ValueError(f"unknown model_family={model_family}")

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols + bin_cols),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop")
    return preprocessor


def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def benchmark_inference(model_predict_proba_fn, X_input, n_repeat=INFER_REPEAT, batch_size=2048):
    # Record per-sample inference latency (milliseconds)
    n = len(X_input)
    idx = np.arange(n)
    latencies = []
    for _ in range(n_repeat):
        np.random.shuffle(idx)
        t0 = time.perf_counter()
        for i in range(0, n, batch_size):
            batch_idx = idx[i:i+batch_size]
            _ = model_predict_proba_fn(X_input.iloc[batch_idx])
        t1 = time.perf_counter()
        elapsed = (t1 - t0) / n * 1000.0
        latencies.append(elapsed)
    return float(np.mean(latencies)), float(np.std(latencies))


def estimate_model_size_mb(model_obj):
    # Estimate model size via temporary serialization
    import tempfile
    import joblib
    with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tmp:
        joblib.dump(model_obj, tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
    try:
        os.remove(tmp.name)
    except Exception:
        pass
    return float(size_mb)


def print_seed_metrics(model_name, seed_results):
    print(f"\n[SUMMARY] {model_name}")
    for m in ["accuracy", "auc_roc", "f1", "train_seconds", "infer_ms_per_sample", "model_size_mb"]:
        vals = [r[m] for r in seed_results]
        print(f"  - {m:20s}: mean={np.mean(vals):.6f}, std={np.std(vals):.6f}")

In [ ]:
# ===== 2.2 Result Containers, Logging Helpers, and Checkpoint Resume Utilities =====
all_results = {}
best_models = {}
best_preprocessors = {}

def optuna_log_callback(study, trial):
    print(
        f"[Optuna] Trial {trial.number:03d} | value={trial.value:.6f} | "
        f"params={trial.params}"
    )

# ===== checkpoint / resume =====
CKPT_ROOT = RESULT_DIR / "checkpoints"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

def _safe_model_name(model_name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in model_name)

def get_seed_ckpt_paths(model_name: str, seed: int):
    seed_dir = CKPT_ROOT / f"{_safe_model_name(model_name)}_seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dir": seed_dir,
        "meta": seed_dir / "meta.json",
        "model_joblib": seed_dir / "model.joblib",
        "bundle_joblib": seed_dir / "bundle.joblib",
        "tabnet_zip": seed_dir / "tabnet_model.zip",
        "ft_state": seed_dir / "ft_state.pt",
    }

def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

def save_seed_meta(model_name: str, seed: int, payload: dict):
    paths = get_seed_ckpt_paths(model_name, seed)
    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=_json_default)

def load_seed_meta(model_name: str, seed: int):
    paths = get_seed_ckpt_paths(model_name, seed)
    if not paths["meta"].exists():
        return None
    with open(paths["meta"], "r", encoding="utf-8") as f:
        return json.load(f)

def compact_deep_bundle(bundle: dict):
    return {
        "num_imputer": bundle["num_imputer"],
        "cat_encoder": bundle["cat_encoder"],
        "cat_dims": bundle["cat_dims"],
        "n_num": bundle["n_num"],
        "n_cat": bundle["n_cat"],
    }

def build_deep_infer_bundle_from_compact(compact_bundle: dict, Xte_df: pd.DataFrame):
    numbin_cols = num_cols + bin_cols
    Xte_num = compact_bundle["num_imputer"].transform(Xte_df[numbin_cols]).astype(np.float32)

    if len(cat_cols) > 0 and compact_bundle.get("cat_encoder") is not None:
        te_cat = Xte_df[cat_cols].astype("string").fillna("MISSING")
        Xte_cat = compact_bundle["cat_encoder"].transform(te_cat).astype(np.int64) + 1
    else:
        Xte_cat = np.zeros((len(Xte_df), 0), dtype=np.int64)

    return {
        "Xte_num": Xte_num,
        "Xte_cat": Xte_cat,
        "cat_dims": compact_bundle["cat_dims"],
        "n_num": compact_bundle["n_num"],
        "n_cat": compact_bundle["n_cat"],
    }

print("[INFO] Utilities + checkpoint manager ready")
print(f"[INFO] Checkpoint root: {CKPT_ROOT}")

[INFO] Utilities + checkpoint manager ready
[INFO] Checkpoint root: artifacts/checkpoints


## 3) Model Training and Tuning (Separate Block per Model)

Notes:
- Each model uses the same Optuna tuning budget (`N_TRIALS`)
- Each model is trained with 3 random seeds on the same data split
- Outputs: validation logs per seed, test metrics, training time, inference latency, and model size

In [ ]:
# ===== 2.3 Unified Tuning and Evaluation Runner for Classical Models (Checkpoint Resume Enabled) =====
def run_classical_experiment(
    model_name,
    model_builder_fn,
    param_suggester_fn,
    model_family,
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    ):
    import joblib

    seed_results = []

    for seed in seeds:
        paths = get_seed_ckpt_paths(model_name, seed)
        cached_meta = load_seed_meta(model_name, seed)
        if cached_meta is not None and paths["model_joblib"].exists():
            cached_metrics = cached_meta.get("metrics", {})
            if "seed" not in cached_metrics:
                cached_metrics["seed"] = int(seed)
            print(f"[RESUME] {model_name} seed={seed} loaded from checkpoint, skip training")
            seed_results.append(cached_metrics)
            continue

        print(f"\n{'='*20} {model_name} | seed={seed} {'='*20}")
        np.random.seed(seed)
        random.seed(seed)

        # FAST_MODE: use subsampling during tuning to reduce training time
        X_train_seed = X_train
        y_train_seed = y_train
        X_valid_seed = X_valid
        y_valid_seed = y_valid

        if MAX_TRAIN_ROWS is not None and len(X_train_seed) > MAX_TRAIN_ROWS:
            idx = X_train_seed.sample(n=MAX_TRAIN_ROWS, random_state=seed).index
            X_train_seed = X_train_seed.loc[idx]
            y_train_seed = y_train_seed.loc[idx]

        if MAX_VALID_ROWS is not None and len(X_valid_seed) > MAX_VALID_ROWS:
            idx = X_valid_seed.sample(n=MAX_VALID_ROWS, random_state=seed).index
            X_valid_seed = X_valid_seed.loc[idx]
            y_valid_seed = y_valid_seed.loc[idx]

        if use_native_cat:
            # Path for CatBoost native categorical support
            Xtr = X_train_seed.copy()
            Xva = X_valid_seed.copy()
            Xte = X_test.copy()

            for c in cat_cols:
                Xtr[c] = Xtr[c].astype("string").fillna("MISSING")
                Xva[c] = Xva[c].astype("string").fillna("MISSING")
                Xte[c] = Xte[c].astype("string").fillna("MISSING")
        else:
            preprocessor = build_sklearn_preprocessor(model_family)
            Xtr = preprocessor.fit_transform(X_train_seed)
            Xva = preprocessor.transform(X_valid_seed)
            Xte = preprocessor.transform(X_test)

        def objective(trial):
            params = param_suggester_fn(trial)
            model = model_builder_fn(params, seed)

            t0 = time.perf_counter()
            if use_native_cat:
                model.fit(
                    Xtr, y_train_seed,
                    eval_set=(Xva, y_valid_seed),
                    cat_features=cat_cols,
                    use_best_model=True,
                    verbose=100
                )
                yva_prob = model.predict_proba(Xva)[:, 1]
            else:
                model.fit(Xtr, y_train_seed)
                yva_prob = model.predict_proba(Xva)[:, 1]
            t1 = time.perf_counter()

            auc = roc_auc_score(y_valid_seed, yva_prob)
            if verbose:
                print(
                    f"[{model_name}|seed={seed}] trial={trial.number} auc={auc:.6f} "
                    f"train_time={t1-t0:.2f}s"
                )
            return auc

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, callbacks=[optuna_log_callback])

        best_params = study.best_trial.params
        print(f"[BEST] {model_name} seed={seed} best_auc(valid)={study.best_value:.6f}")
        print(f"[BEST] params={best_params}")

        final_model = model_builder_fn(best_params, seed)
        t0 = time.perf_counter()
        if use_native_cat:
            final_model.fit(
                Xtr, y_train_seed,
                eval_set=(Xva, y_valid_seed),
                cat_features=cat_cols,
                use_best_model=True,
                verbose=100
            )
            y_prob_test = final_model.predict_proba(Xte)[:, 1]
            infer_mean, infer_std = benchmark_inference(
                lambda x: final_model.predict_proba(x)[:, 1], Xte
            )
            model_size = estimate_model_size_mb(final_model)
            preprocess_obj = None
        else:
            final_model.fit(Xtr, y_train_seed)
            y_prob_test = final_model.predict_proba(Xte)[:, 1]
            infer_mean, infer_std = benchmark_inference(
                lambda x: final_model.predict_proba(preprocessor.transform(x))[:, 1],
                X_test
            )
            try:
                model_size = estimate_model_size_mb((preprocessor, final_model))
            except Exception:
                model_size = np.nan
            preprocess_obj = preprocessor
        t1 = time.perf_counter()

        m = evaluate_predictions(y_test, y_prob_test)
        m["seed"] = int(seed)
        m["train_seconds"] = float(t1 - t0)
        m["infer_ms_per_sample"] = float(infer_mean)
        m["infer_ms_per_sample_std"] = float(infer_std)
        m["model_size_mb"] = float(model_size)
        m["best_valid_auc"] = float(study.best_value)
        m["best_params"] = best_params
        seed_results.append(m)

        # Save checkpoint per seed to support resume after interruption
        joblib.dump({"model": final_model, "preprocessor": preprocess_obj}, paths["model_joblib"])
        save_seed_meta(
            model_name,
            seed,
            {
                "model_name": model_name,
                "seed": int(seed),
                "format": "joblib",
                "metrics": m,
                "best_params": best_params,
            },
        )
        print(f"[CKPT] Saved {model_name} seed={seed} -> {paths['dir']}")

        gc.collect()

    if len(seed_results) == 0:
        raise RuntimeError(f"No seed results found for model {model_name}")

    all_results[model_name] = seed_results

    # Deserialize from best-seed checkpoint so post-restart prediction can continue
    best_seed = int(max(seed_results, key=lambda r: r["auc_roc"])["seed"])
    best_paths = get_seed_ckpt_paths(model_name, best_seed)
    best_pack = joblib.load(best_paths["model_joblib"])

    best_models[model_name] = best_pack["model"]
    best_preprocessors[model_name] = best_pack["preprocessor"]

    print(f"[RESUME] {model_name} best seed selected: {best_seed}")
    print_seed_metrics(model_name, seed_results)

In [ ]:
# ===== 3.1 Logistic Regression =====
def logistic_builder(params, seed):
    return LogisticRegression(
        C=params["C"],
        penalty=params["penalty"],
        solver="saga",
        max_iter=3000,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )

def logistic_suggester(trial):
    return {
        "C": trial.suggest_float("C", 1e-3, 30.0, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
    }

run_classical_experiment(
    model_name="LogisticRegression",
    model_builder_fn=logistic_builder,
    param_suggester_fn=logistic_suggester,
    model_family="linear",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== LogisticRegression | seed=123 ====================


[I 2026-04-05 04:01:12,974] A new study created in memory with name: no-name-6a24216a-23b7-4254-966f-a9507212bdd3
[I 2026-04-05 04:08:54,962] Trial 0 finished with value: 0.622586929039531 and parameters: {'C': 0.3450922232156312, 'penalty': 'l2'}. Best is trial 0 with value: 0.622586929039531.


[LogisticRegression|seed=123] trial=0 auc=0.622587 train_time=461.96s
[Optuna] Trial 000 | value=0.622587 | params={'C': 0.3450922232156312, 'penalty': 'l2'}


[I 2026-04-05 04:24:57,035] Trial 1 finished with value: 0.6227870935335877 and parameters: {'C': 0.3015430440745738, 'penalty': 'l1'}. Best is trial 1 with value: 0.6227870935335877.


[LogisticRegression|seed=123] trial=1 auc=0.622787 train_time=962.04s
[Optuna] Trial 001 | value=0.622787 | params={'C': 0.3015430440745738, 'penalty': 'l1'}


...

[LogisticRegression|seed=789] trial=2 auc=0.623312 train_time=70.91s
[Optuna] Trial 002 | value=0.623312 | params={'C': 0.014376195292490474, 'penalty': 'l2'}


[I 2026-04-05 07:17:03,749] Trial 3 finished with value: 0.6226100252518052 and parameters: {'C': 1.1568615941717317, 'penalty': 'l1'}. Best is trial 2 with value: 0.6233117100814108.


[LogisticRegression|seed=789] trial=3 auc=0.622610 train_time=808.86s
[Optuna] Trial 003 | value=0.622610 | params={'C': 1.1568615941717317, 'penalty': 'l1'}


[I 2026-04-05 07:19:19,302] Trial 4 finished with value: 0.6229571053649566 and parameters: {'C': 0.03420554062243378, 'penalty': 'l2'}. Best is trial 2 with value: 0.6233117100814108.


[LogisticRegression|seed=789] trial=4 auc=0.622957 train_time=135.53s
[Optuna] Trial 004 | value=0.622957 | params={'C': 0.03420554062243378, 'penalty': 'l2'}
[BEST] LogisticRegression seed=789 best_auc(valid)=0.623312
[BEST] params={'C': 0.014376195292490474, 'penalty': 'l2'}
[CKPT] Saved LogisticRegression seed=789 -> artifacts/checkpoints/LogisticRegression_seed789
[RESUME] LogisticRegression best seed selected: 456

[SUMMARY] LogisticRegression
  - accuracy            : mean=0.620059, std=0.000699
  - auc_roc             : mean=0.625624, std=0.001916
  - f1                  : mean=0.095956, std=0.000652
  - train_seconds       : mean=816.116632, std=859.861846
  - infer_ms_per_sample : mean=0.006752, std=0.000122
  - model_size_mb       : mean=0.011026, std=0.000000


In [ ]:
# ===== 3.2 Random Forest =====
def rf_builder(params, seed):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=seed,
    )

def rf_suggester(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
        "max_depth": trial.suggest_int("max_depth", 4, 24),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }

run_classical_experiment(
    model_name="RandomForest",
    model_builder_fn=rf_builder,
    param_suggester_fn=rf_suggester,
    model_family="tree",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== RandomForest | seed=123 ====================


[I 2026-04-05 07:20:38,256] A new study created in memory with name: no-name-cf304048-82f1-4376-ac31-4ac7747073a1
[I 2026-04-05 07:21:25,685] Trial 0 finished with value: 0.5962209280270669 and parameters: {'n_estimators': 300, 'max_depth': 14, 'min_samples_split': 15, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.5962209280270669.


[RandomForest|seed=123] trial=0 auc=0.596221 train_time=47.40s
[Optuna] Trial 000 | value=0.596221 | params={'n_estimators': 300, 'max_depth': 14, 'min_samples_split': 15, 'min_samples_leaf': 7, 'max_features': 'log2'}


[I 2026-04-05 07:22:05,538] Trial 1 finished with value: 0.614548440538925 and parameters: {'n_estimators': 400, 'max_depth': 4, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.614548440538925.


[RandomForest|seed=123] trial=1 auc=0.614548 train_time=39.82s
[Optuna] Trial 001 | value=0.614548 | params={'n_estimators': 400, 'max_depth': 4, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 'sqrt'}


...

[RandomForest|seed=789] trial=2 auc=0.620645 train_time=127.74s
[Optuna] Trial 002 | value=0.620645 | params={'n_estimators': 1100, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 10, 'max_features': 'log2'}


[I 2026-04-05 08:11:04,481] Trial 3 finished with value: 0.6154984473237262 and parameters: {'n_estimators': 200, 'max_depth': 4, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 'log2'}. Best is trial 2 with value: 0.6206450549665339.


[RandomForest|seed=789] trial=3 auc=0.615498 train_time=18.12s
[Optuna] Trial 003 | value=0.615498 | params={'n_estimators': 200, 'max_depth': 4, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 'log2'}


[I 2026-04-05 08:13:29,016] Trial 4 finished with value: 0.6191855893394683 and parameters: {'n_estimators': 200, 'max_depth': 7, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 2 with value: 0.6206450549665339.


[RandomForest|seed=789] trial=4 auc=0.619186 train_time=144.50s
[Optuna] Trial 004 | value=0.619186 | params={'n_estimators': 200, 'max_depth': 7, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': None}
[BEST] RandomForest seed=789 best_auc(valid)=0.620645
[BEST] params={'n_estimators': 1100, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 10, 'max_features': 'log2'}
[CKPT] Saved RandomForest seed=789 -> artifacts/checkpoints/RandomForest_seed789
[RESUME] RandomForest best seed selected: 789

[SUMMARY] RandomForest
  - accuracy            : mean=0.664457, std=0.053086
  - auc_roc             : mean=0.616941, std=0.001500
  - f1                  : mean=0.096822, std=0.001748
  - train_seconds       : mean=177.385270, std=46.686830
  - infer_ms_per_sample : mean=0.131132, std=0.032507
  - model_size_mb       : mean=31.085119, std=26.741593


In [ ]:
# ===== 3.3 XGBoost =====
def xgb_builder(params, seed):
    xgb_common = dict(
        objective="binary:logistic",
        eval_metric="auc",
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        n_estimators=params["n_estimators"],
        random_state=seed,
        n_jobs=-1,
    )
    if USE_GPU:
        # XGBoost 2.x recommends device='cuda' + hist
        return XGBClassifier(tree_method="hist", device="cuda", **xgb_common)
    return XGBClassifier(tree_method="hist", **xgb_common)

def xgb_suggester(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1400, step=100),
    }

run_classical_experiment(
    model_name="XGBoost",
    model_builder_fn=xgb_builder,
    param_suggester_fn=xgb_suggester,
    model_family="tree",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== XGBoost | seed=123 ====================


[I 2026-04-05 08:17:23,263] A new study created in memory with name: no-name-1c1ad658-f455-4c75-b0da-7390fed5a3e8
[I 2026-04-05 08:17:30,450] Trial 0 finished with value: 0.5801933260863534 and parameters: {'learning_rate': 0.11173563190654363, 'max_depth': 6, 'min_child_weight': 4.784472615172413, 'subsample': 0.6344487326037691, 'colsample_bytree': 0.771510036023904, 'reg_alpha': 0.0072062010845638635, 'reg_lambda': 0.4608317091104452, 'n_estimators': 1300}. Best is trial 0 with value: 0.5801933260863534.


[XGBoost|seed=123] trial=0 auc=0.580193 train_time=7.14s
[Optuna] Trial 000 | value=0.580193 | params={'learning_rate': 0.11173563190654363, 'max_depth': 6, 'min_child_weight': 4.784472615172413, 'subsample': 0.6344487326037691, 'colsample_bytree': 0.771510036023904, 'reg_alpha': 0.0072062010845638635, 'reg_lambda': 0.4608317091104452, 'n_estimators': 1300}


[I 2026-04-05 08:17:32,348] Trial 1 finished with value: 0.6262655086795786 and parameters: {'learning_rate': 0.19477181057696416, 'max_depth': 3, 'min_child_weight': 0.24432548003700166, 'subsample': 0.8743921662054399, 'colsample_bytree': 0.7988246277710186, 'reg_alpha': 4.033231047390516e-05, 'reg_lambda': 0.002758223855977828, 'n_estimators': 500}. Best is trial 1 with value: 0.6262655086795786.


[XGBoost|seed=123] trial=1 auc=0.626266 train_time=1.86s
[Optuna] Trial 001 | value=0.626266 | params={'learning_rate': 0.19477181057696416, 'max_depth': 3, 'min_child_weight': 0.24432548003700166, 'subsample': 0.8743921662054399, 'colsample_bytree': 0.7988246277710186, 'reg_alpha': 4.033231047390516e-05, 'reg_lambda': 0.002758223855977828, 'n_estimators': 500}


...

[XGBoost|seed=789] trial=2 auc=0.574071 train_time=4.51s
[Optuna] Trial 002 | value=0.574071 | params={'learning_rate': 0.1334427232368235, 'max_depth': 7, 'min_child_weight': 0.10931478305228041, 'subsample': 0.8551461527332738, 'colsample_bytree': 0.5464767112312398, 'reg_alpha': 5.939661176861022e-08, 'reg_lambda': 2.008958002335519e-06, 'n_estimators': 700}


[I 2026-04-05 08:18:44,864] Trial 3 finished with value: 0.6160075776552724 and parameters: {'learning_rate': 0.04248856382780391, 'max_depth': 7, 'min_child_weight': 16.88108135894345, 'subsample': 0.8057991119645812, 'colsample_bytree': 0.8484712718112042, 'reg_alpha': 0.2013427510507727, 'reg_lambda': 5.643805895891701, 'n_estimators': 1300}. Best is trial 0 with value: 0.6348707494972641.


[XGBoost|seed=789] trial=3 auc=0.616008 train_time=7.72s
[Optuna] Trial 003 | value=0.616008 | params={'learning_rate': 0.04248856382780391, 'max_depth': 7, 'min_child_weight': 16.88108135894345, 'subsample': 0.8057991119645812, 'colsample_bytree': 0.8484712718112042, 'reg_alpha': 0.2013427510507727, 'reg_lambda': 5.643805895891701, 'n_estimators': 1300}


[I 2026-04-05 08:18:49,229] Trial 4 finished with value: 0.6293907959004177 and parameters: {'learning_rate': 0.014707349542611832, 'max_depth': 9, 'min_child_weight': 0.39499484550107444, 'subsample': 0.6298867452398829, 'colsample_bytree': 0.8524183916826701, 'reg_alpha': 2.1649693345769293e-06, 'reg_lambda': 0.010557679586468777, 'n_estimators': 400}. Best is trial 0 with value: 0.6348707494972641.


[XGBoost|seed=789] trial=4 auc=0.629391 train_time=4.33s
[Optuna] Trial 004 | value=0.629391 | params={'learning_rate': 0.014707349542611832, 'max_depth': 9, 'min_child_weight': 0.39499484550107444, 'subsample': 0.6298867452398829, 'colsample_bytree': 0.8524183916826701, 'reg_alpha': 2.1649693345769293e-06, 'reg_lambda': 0.010557679586468777, 'n_estimators': 400}
[BEST] XGBoost seed=789 best_auc(valid)=0.634871
[BEST] params={'learning_rate': 0.022127403832621583, 'max_depth': 3, 'min_child_weight': 1.7790267814124234, 'subsample': 0.7945567382721752, 'colsample_bytree': 0.8696824869009516, 'reg_alpha': 0.0014139104597015485, 'reg_lambda': 7.028978444644654e-05, 'n_estimators': 600}
[CKPT] Saved XGBoost seed=789 -> artifacts/checkpoints/XGBoost_seed789
[RESUME] XGBoost best seed selected: 456

[SUMMARY] XGBoost
  - accuracy            : mean=0.963551, std=0.000000
  - auc_roc             : mean=0.634833, std=0.000335
  - f1                  : mean=0.000000, std=0.000000
  - train_secon

In [ ]:
# ===== 3.4 LightGBM =====
def lgbm_builder(params, seed):
    return LGBMClassifier(
        objective="binary",
        metric="auc",
        learning_rate=params["learning_rate"],
        n_estimators=params["n_estimators"],
        num_leaves=params["num_leaves"],
        max_depth=params["max_depth"],
        min_child_samples=params["min_child_samples"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        random_state=seed,
        n_jobs=-1,
        verbosity=1,
        device_type="gpu" if USE_GPU else "cpu",
    )

def lgbm_suggester(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1800, step=100),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

run_classical_experiment(
    model_name="LightGBM",
    model_builder_fn=lgbm_builder,
    param_suggester_fn=lgbm_suggester,
    model_family="tree",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== LightGBM | seed=123 ====================


[I 2026-04-05 08:18:58,673] A new study created in memory with name: no-name-769f8ea3-25c5-4578-94b4-760c94944ddb


[LightGBM] [Info] Number of positive: 13016, number of negative: 344110
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1609
[LightGBM] [Info] Number of data points in the train set: 357126, number of used features: 58
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 35 dense feature groups (12.26 MB) transferred to GPU in 0.011612 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274782
[LightGBM] [Info] Start training from score -3.274782


[I 2026-04-05 08:19:22,490] Trial 0 finished with value: 0.6308098737969112 and parameters: {'learning_rate': 0.03439031558660186, 'n_estimators': 1200, 'num_leaves': 27, 'max_depth': 16, 'min_child_samples': 100, 'subsample': 0.8046998006115287, 'colsample_bytree': 0.5491797894669113, 'reg_alpha': 7.747445968206635, 'reg_lambda': 9.108783169342993e-06}. Best is trial 0 with value: 0.6308098737969112.


[LightGBM|seed=123] trial=0 auc=0.630810 train_time=23.79s
[Optuna] Trial 000 | value=0.630810 | params={'learning_rate': 0.03439031558660186, 'n_estimators': 1200, 'num_leaves': 27, 'max_depth': 16, 'min_child_samples': 100, 'subsample': 0.8046998006115287, 'colsample_bytree': 0.5491797894669113, 'reg_alpha': 7.747445968206635, 'reg_lambda': 9.108783169342993e-06}
[LightGBM] [Info] Number of positive: 13016, number of negative: 344110
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1609
[LightGBM] [Info] Number of data points in the train set: 357126, number of used features: 58
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 35 dense feature groups (12.26 MB) transferred to GPU in 0.011596 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pav

...

[LightGBM|seed=789] trial=2 auc=0.606009 train_time=15.53s
[Optuna] Trial 002 | value=0.606009 | params={'learning_rate': 0.0636295835032485, 'n_estimators': 400, 'num_leaves': 208, 'max_depth': 0, 'min_child_samples': 65, 'subsample': 0.5121067527647679, 'colsample_bytree': 0.6300222930162904, 'reg_alpha': 3.611918735610124e-07, 'reg_lambda': 0.0007168519183077282}
[LightGBM] [Info] Number of positive: 13016, number of negative: 344110
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1613
[LightGBM] [Info] Number of data points in the train set: 357126, number of used features: 58
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 35 dense feature groups (12.26 MB) transferred to GPU in 0.011471 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pa

[I 2026-04-05 08:25:29,862] Trial 3 finished with value: 0.6205491700215883 and parameters: {'learning_rate': 0.023197464550952545, 'n_estimators': 900, 'num_leaves': 222, 'max_depth': 13, 'min_child_samples': 34, 'subsample': 0.7931608352777721, 'colsample_bytree': 0.708901341436224, 'reg_alpha': 1.2333330717536986, 'reg_lambda': 1.7082400761897393e-07}. Best is trial 1 with value: 0.6344901904248244.


[LightGBM|seed=789] trial=3 auc=0.620549 train_time=38.40s
[Optuna] Trial 003 | value=0.620549 | params={'learning_rate': 0.023197464550952545, 'n_estimators': 900, 'num_leaves': 222, 'max_depth': 13, 'min_child_samples': 34, 'subsample': 0.7931608352777721, 'colsample_bytree': 0.708901341436224, 'reg_alpha': 1.2333330717536986, 'reg_lambda': 1.7082400761897393e-07}
[LightGBM] [Info] Number of positive: 13016, number of negative: 344110
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1613
[LightGBM] [Info] Number of data points in the train set: 357126, number of used features: 58
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 35 dense feature groups (12.26 MB) transferred to GPU in 0.012529 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pa

[I 2026-04-05 08:26:33,389] Trial 4 finished with value: 0.5985414212514119 and parameters: {'learning_rate': 0.026268280465006115, 'n_estimators': 1800, 'num_leaves': 191, 'max_depth': 13, 'min_child_samples': 77, 'subsample': 0.9467357777803387, 'colsample_bytree': 0.7368093051235503, 'reg_alpha': 4.863579058459068e-07, 'reg_lambda': 5.432712995150697e-08}. Best is trial 1 with value: 0.6344901904248244.


[LightGBM|seed=789] trial=4 auc=0.598541 train_time=63.49s
[Optuna] Trial 004 | value=0.598541 | params={'learning_rate': 0.026268280465006115, 'n_estimators': 1800, 'num_leaves': 191, 'max_depth': 13, 'min_child_samples': 77, 'subsample': 0.9467357777803387, 'colsample_bytree': 0.7368093051235503, 'reg_alpha': 4.863579058459068e-07, 'reg_lambda': 5.432712995150697e-08}
[BEST] LightGBM seed=789 best_auc(valid)=0.634490
[BEST] params={'learning_rate': 0.0361939152362532, 'n_estimators': 800, 'num_leaves': 56, 'max_depth': 3, 'min_child_samples': 24, 'subsample': 0.9072226176812946, 'colsample_bytree': 0.8909437324352822, 'reg_alpha': 0.0011836728969323266, 'reg_lambda': 0.0038699301510127988}
[LightGBM] [Info] Number of positive: 13016, number of negative: 344110
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1613
[LightGBM] [Info] Number of data points in the train set: 357126, number of used features: 58
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: N

In [ ]:
# ===== 3.5 CatBoost =====
def cat_builder(params, seed):
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],
        iterations=params["iterations"],
        random_seed=seed,
        verbose=100,
        task_type="GPU" if USE_GPU else "CPU",
        train_dir=str(CATBOOST_TRAIN_DIR),
    )

def cat_suggester(trial):
    return {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        "iterations": trial.suggest_int("iterations", 300, 2000, step=100),
    }

run_classical_experiment(
    model_name="CatBoost",
    model_builder_fn=cat_builder,
    param_suggester_fn=cat_suggester,
    model_family="tree",
    use_native_cat=True,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== CatBoost | seed=123 ====================


[I 2026-04-05 08:26:53,872] A new study created in memory with name: no-name-169773fc-aa8c-4a62-9eb1-25fc89529fc5
Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5568849	best: 0.5568849 (0)	total: 118ms	remaining: 1m 22s
100:	test: 0.6318208	best: 0.6318208 (100)	total: 7.91s	remaining: 46.9s
200:	test: 0.6335561	best: 0.6335561 (200)	total: 15.8s	remaining: 39.2s
300:	test: 0.6344244	best: 0.6344373 (285)	total: 23.9s	remaining: 31.6s
400:	test: 0.6342730	best: 0.6344771 (320)	total: 32.1s	remaining: 24s
500:	test: 0.6339919	best: 0.6344771 (320)	total: 40.4s	remaining: 16s
600:	test: 0.6328574	best: 0.6344771 (320)	total: 48.8s	remaining: 8.04s
699:	test: 0.6314823	best: 0.6344771 (320)	total: 57.2s	remaining: 0us
bestTest = 0.6344771385
bestIteration = 320
Shrink model to first 321 iterations.


[I 2026-04-05 08:27:55,075] Trial 0 finished with value: 0.6344771303656871 and parameters: {'depth': 8, 'learning_rate': 0.08432113568876762, 'l2_leaf_reg': 15.554906428804477, 'iterations': 700}. Best is trial 0 with value: 0.6344771303656871.


[CatBoost|seed=123] trial=0 auc=0.634477 train_time=61.17s
[Optuna] Trial 000 | value=0.634477 | params={'depth': 8, 'learning_rate': 0.08432113568876762, 'l2_leaf_reg': 15.554906428804477, 'iterations': 700}


...

0:	test: 0.5606989	best: 0.5606989 (0)	total: 164ms	remaining: 1m 37s
100:	test: 0.6305565	best: 0.6308137 (95)	total: 10.1s	remaining: 50s
200:	test: 0.6302477	best: 0.6309150 (105)	total: 20.5s	remaining: 40.7s
300:	test: 0.6275191	best: 0.6309150 (105)	total: 31.1s	remaining: 30.9s
400:	test: 0.6220148	best: 0.6309150 (105)	total: 41.8s	remaining: 20.7s
500:	test: 0.6181541	best: 0.6309150 (105)	total: 52.5s	remaining: 10.4s
599:	test: 0.6149572	best: 0.6309150 (105)	total: 1m 3s	remaining: 0us
bestTest = 0.6309149861
bestIteration = 105
Shrink model to first 106 iterations.


[I 2026-04-05 08:54:45,166] Trial 4 finished with value: 0.630915009282206 and parameters: {'depth': 9, 'learning_rate': 0.14166609037852596, 'l2_leaf_reg': 11.851226387059095, 'iterations': 600}. Best is trial 0 with value: 0.6352821182538283.


[CatBoost|seed=789] trial=4 auc=0.630915 train_time=66.80s
[Optuna] Trial 004 | value=0.630915 | params={'depth': 9, 'learning_rate': 0.14166609037852596, 'l2_leaf_reg': 11.851226387059095, 'iterations': 600}
[BEST] CatBoost seed=789 best_auc(valid)=0.635282
[BEST] params={'depth': 10, 'learning_rate': 0.012459911629562195, 'l2_leaf_reg': 11.896767510906313, 'iterations': 1000}


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5607904	best: 0.5607904 (0)	total: 204ms	remaining: 3m 23s
100:	test: 0.6239571	best: 0.6239571 (100)	total: 13.2s	remaining: 1m 57s
200:	test: 0.6278968	best: 0.6278968 (200)	total: 26.7s	remaining: 1m 46s
300:	test: 0.6307547	best: 0.6307547 (300)	total: 40s	remaining: 1m 32s
400:	test: 0.6323106	best: 0.6323106 (400)	total: 53.6s	remaining: 1m 20s
500:	test: 0.6332186	best: 0.6332186 (500)	total: 1m 6s	remaining: 1m 6s
600:	test: 0.6338102	best: 0.6338102 (600)	total: 1m 20s	remaining: 53.1s
700:	test: 0.6341583	best: 0.6341583 (700)	total: 1m 33s	remaining: 39.9s
800:	test: 0.6343452	best: 0.6343525 (760)	total: 1m 47s	remaining: 26.6s
900:	test: 0.6344573	best: 0.6344573 (900)	total: 2m	remaining: 13.3s
999:	test: 0.6349146	best: 0.6349146 (999)	total: 2m 14s	remaining: 0us
bestTest = 0.6349146068
bestIteration = 999
[CKPT] Saved CatBoost seed=789 -> artifacts/checkpoints/CatBoost_seed789
[RESUME] CatBoost best seed selected: 456

[SUMMARY] CatBoost
  - accuracy        

In [ ]:
# ===== 3.6 Deep Model Utilities: TabNet / FT-Transformer Preprocessing and Training =====
# FT-Transformer implementation package
try:
    from tab_transformer_pytorch import FTTransformer
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tab-transformer-pytorch"])
    from tab_transformer_pytorch import FTTransformer

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
DEVICE = "cpu" if FT_FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Torch device: {DEVICE}")

def prepare_deep_inputs(Xtr_df, Xva_df, Xte_df):
    """Return encoded data that can be used by TabNet/FT-Transformer."""
    numbin_cols = num_cols + bin_cols
    # Numeric features: median imputation
    num_imputer = SimpleImputer(strategy="median")
    Xtr_num = num_imputer.fit_transform(Xtr_df[numbin_cols])
    Xva_num = num_imputer.transform(Xva_df[numbin_cols])
    Xte_num = num_imputer.transform(Xte_df[numbin_cols])

    # Categorical features: cast to string + missing placeholder + ordinal encoding (unknown -> -1, then +1)
    if len(cat_cols) > 0:
        tr_cat = Xtr_df[cat_cols].astype("string").fillna("MISSING")
        va_cat = Xva_df[cat_cols].astype("string").fillna("MISSING")
        te_cat = Xte_df[cat_cols].astype("string").fillna("MISSING")
        cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        Xtr_cat = cat_encoder.fit_transform(tr_cat).astype(np.int64) + 1
        Xva_cat = cat_encoder.transform(va_cat).astype(np.int64) + 1
        Xte_cat = cat_encoder.transform(te_cat).astype(np.int64) + 1
        cat_dims = [int(Xtr_cat[:, i].max() + 1) for i in range(Xtr_cat.shape[1])]
    else:
        Xtr_cat = np.zeros((len(Xtr_df), 0), dtype=np.int64)
        Xva_cat = np.zeros((len(Xva_df), 0), dtype=np.int64)
        Xte_cat = np.zeros((len(Xte_df), 0), dtype=np.int64)
        cat_dims = []
        cat_encoder = None

    bundle = {
        "Xtr_num": Xtr_num.astype(np.float32),
        "Xva_num": Xva_num.astype(np.float32),
        "Xte_num": Xte_num.astype(np.float32),
        "Xtr_cat": Xtr_cat,
        "Xva_cat": Xva_cat,
        "Xte_cat": Xte_cat,
        "cat_dims": cat_dims,
        "num_imputer": num_imputer,
        "cat_encoder": cat_encoder,
        "n_num": len(numbin_cols),
        "n_cat": len(cat_cols),
    }
    return bundle

def fit_fttransformer_once(bundle, ytr, yva, params, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = FTTransformer(
        categories=tuple(bundle["cat_dims"]),
        num_continuous=bundle["n_num"],
        dim=params["dim"],
        dim_out=1,
        depth=params["depth"],
        heads=params["heads"],
        attn_dropout=params["attn_dropout"],
        ff_dropout=params["ff_dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()
    xtr_cat = torch.tensor(bundle["Xtr_cat"], dtype=torch.long)
    xtr_num = torch.tensor(bundle["Xtr_num"], dtype=torch.float32)
    ytr_t = torch.tensor(np.asarray(ytr).reshape(-1, 1), dtype=torch.float32)
    train_ds = TensorDataset(xtr_cat, xtr_num, ytr_t)
    train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
    xva_cat = torch.tensor(bundle["Xva_cat"], dtype=torch.long, device=DEVICE)
    xva_num = torch.tensor(bundle["Xva_num"], dtype=torch.float32, device=DEVICE)
    yva_np = np.asarray(yva).astype(int)
    best_auc = -np.inf
    best_state = None
    no_improve = 0

    for epoch in range(1, params["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for bc, bn, by in train_loader:
            bc = bc.to(DEVICE)
            bn = bn.to(DEVICE)
            by = by.to(DEVICE)

            optimizer.zero_grad()
            logits = model(bc, bn)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(by)
        model.eval()
        with torch.no_grad():
            logits_va = model(xva_cat, xva_num).squeeze(1)
            prob_va = torch.sigmoid(logits_va).detach().cpu().numpy()
        auc_va = roc_auc_score(yva_np, prob_va)
        avg_loss = running_loss / len(train_ds)

        print(f"[FTTransformer|seed={seed}] epoch={epoch:03d} loss={avg_loss:.6f} valid_auc={auc_va:.6f}")

        if auc_va > best_auc:
            best_auc = auc_va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= params["patience"]:
            print(f"[FTTransformer|seed={seed}] Early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, float(best_auc)

def ft_predict_proba(model, X_cat, X_num, batch_size=2048):
    model.eval()
    probs = []
    n = len(X_num)
    with torch.no_grad():
        for i in range(0, n, batch_size):
            bc = torch.tensor(X_cat[i:i+batch_size], dtype=torch.long, device=DEVICE)
            bn = torch.tensor(X_num[i:i+batch_size], dtype=torch.float32, device=DEVICE)
            logits = model(bc, bn).squeeze(1)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)

[INFO] Torch device: cuda


In [ ]:
# ===== 3.6 TabNet (checkpoint resume supported) =====
def run_tabnet_experiment(n_trials=N_TRIALS, seeds=SEEDS):
    import joblib
    model_name = "TabNet"
    seed_results = []

    for seed in seeds:
        paths = get_seed_ckpt_paths(model_name, seed)
        cached_meta = load_seed_meta(model_name, seed)
        if cached_meta is not None and paths["tabnet_zip"].exists() and paths["bundle_joblib"].exists():
            cached_metrics = cached_meta.get("metrics", {})
            if "seed" not in cached_metrics:
                cached_metrics["seed"] = int(seed)
            print(f"[RESUME] {model_name} seed={seed} loaded from checkpoint, skip training")
            seed_results.append(cached_metrics)
            continue

        print(f"\n{'='*20} {model_name} | seed={seed} {'='*20}")
        np.random.seed(seed)
        random.seed(seed)
        torch.manual_seed(seed)
        X_train_seed = X_train
        y_train_seed = y_train
        X_valid_seed = X_valid
        y_valid_seed = y_valid

        if MAX_TRAIN_ROWS is not None and len(X_train_seed) > MAX_TRAIN_ROWS:
            idx = X_train_seed.sample(n=MAX_TRAIN_ROWS, random_state=seed).index
            X_train_seed = X_train_seed.loc[idx]
            y_train_seed = y_train_seed.loc[idx]

        if MAX_VALID_ROWS is not None and len(X_valid_seed) > MAX_VALID_ROWS:
            idx = X_valid_seed.sample(n=MAX_VALID_ROWS, random_state=seed).index
            X_valid_seed = X_valid_seed.loc[idx]
            y_valid_seed = y_valid_seed.loc[idx]

        bundle = prepare_deep_inputs(X_train_seed, X_valid_seed, X_test)
        Xtr_tab = np.hstack([bundle["Xtr_num"], bundle["Xtr_cat"]]).astype(np.float32)
        Xva_tab = np.hstack([bundle["Xva_num"], bundle["Xva_cat"]]).astype(np.float32)
        Xte_tab = np.hstack([bundle["Xte_num"], bundle["Xte_cat"]]).astype(np.float32)
        cat_idxs = list(range(bundle["n_num"], bundle["n_num"] + bundle["n_cat"]))
        cat_dims = bundle["cat_dims"]
        ytr_np = y_train_seed.values
        yva_np = y_valid_seed.values
        yte_np = y_test.values

        def objective(trial):
            params = {
                "n_d": trial.suggest_int("n_d", 8, 64, step=8),
                "n_a": trial.suggest_int("n_a", 8, 64, step=8),
                "n_steps": trial.suggest_int("n_steps", 3, 8),
                "gamma": trial.suggest_float("gamma", 1.0, 2.0),
                "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-7, 1e-2, log=True),
                "lr": trial.suggest_float("lr", 1e-4, 5e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [1024, 2048, 4096]),
                "virtual_batch_size": trial.suggest_categorical("virtual_batch_size", [128, 256, 512]),
            }

            model = TabNetClassifier(
                n_d=params["n_d"],
                n_a=params["n_a"],
                n_steps=params["n_steps"],
                gamma=params["gamma"],
                lambda_sparse=params["lambda_sparse"],
                cat_idxs=cat_idxs,
                cat_dims=cat_dims,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=params["lr"]),
                seed=seed,
                verbose=10,
                device_name="cuda" if USE_GPU else "cpu",
            )

            t0 = time.perf_counter()
            model.fit(
                X_train=Xtr_tab, y_train=ytr_np,
                eval_set=[(Xva_tab, yva_np)],
                eval_name=["valid"],
                eval_metric=["auc"],
                max_epochs=TABNET_MAX_EPOCHS,
                patience=min(12, max(5, TABNET_MAX_EPOCHS // 4)),
                batch_size=params["batch_size"],
                virtual_batch_size=params["virtual_batch_size"],
                drop_last=False,
            )
            t1 = time.perf_counter()
            yva_prob = model.predict_proba(Xva_tab)[:, 1]
            auc = roc_auc_score(yva_np, yva_prob)
            print(f"[TabNet|seed={seed}] trial={trial.number} auc={auc:.6f} train_time={t1-t0:.2f}s")
            return auc

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, callbacks=[optuna_log_callback])
        best_params = study.best_trial.params
        print(f"[BEST] TabNet seed={seed} best_valid_auc={study.best_value:.6f}")
        print(f"[BEST] params={best_params}")

        final_model = TabNetClassifier(
            n_d=best_params["n_d"],
            n_a=best_params["n_a"],
            n_steps=best_params["n_steps"],
            gamma=best_params["gamma"],
            lambda_sparse=best_params["lambda_sparse"],
            cat_idxs=cat_idxs,
            cat_dims=cat_dims,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=best_params["lr"]),
            seed=seed,
            verbose=10,
            device_name="cuda" if USE_GPU else "cpu",
        )

        t0 = time.perf_counter()
        final_model.fit(
            X_train=Xtr_tab, y_train=ytr_np,
            eval_set=[(Xva_tab, yva_np)],
            eval_name=["valid"],
            eval_metric=["auc"],
            max_epochs=TABNET_MAX_EPOCHS,
            patience=min(12, max(5, TABNET_MAX_EPOCHS // 4)),
            batch_size=best_params["batch_size"],
            virtual_batch_size=best_params["virtual_batch_size"],
            drop_last=False,
        )
        y_prob_test = final_model.predict_proba(Xte_tab)[:, 1]
        t1 = time.perf_counter()

        m = evaluate_predictions(yte_np, y_prob_test)
        m["seed"] = int(seed)
        m["train_seconds"] = float(t1 - t0)

        # Inference latency benchmarking
        lat_list = []
        for _ in range(INFER_REPEAT):
            start = time.perf_counter()
            _ = final_model.predict_proba(Xte_tab)[:, 1]
            end = time.perf_counter()
            lat_list.append((end - start) / len(Xte_tab) * 1000)
        m["infer_ms_per_sample"] = float(np.mean(lat_list))
        m["infer_ms_per_sample_std"] = float(np.std(lat_list))

        # Save model checkpoint files
        save_prefix = str(paths["dir"] / "tabnet_model")
        saved_path = final_model.save_model(save_prefix)
        zip_path = Path(saved_path if str(saved_path).endswith(".zip") else f"{saved_path}.zip")
        if not zip_path.exists():
            zip_path = paths["tabnet_zip"]

        # Compatibility fallback: ensure standard path exists
        if zip_path != paths["tabnet_zip"] and zip_path.exists():
            import shutil
            shutil.copy2(zip_path, paths["tabnet_zip"] )
            zip_path = paths["tabnet_zip"]

        m["model_size_mb"] = float(os.path.getsize(zip_path) / (1024 * 1024))
        m["best_valid_auc"] = float(study.best_value)
        m["best_params"] = best_params
        seed_results.append(m)

        compact_bundle = compact_deep_bundle(bundle)
        joblib.dump(compact_bundle, paths["bundle_joblib"])
        save_seed_meta(
            model_name,
            seed,
            {
                "model_name": model_name,
                "seed": int(seed),
                "format": "tabnet",
                "metrics": m,
                "best_params": best_params,
                "model_zip": str(zip_path),
            },
        )
        print(f"[CKPT] Saved {model_name} seed={seed} -> {paths['dir']}")
        gc.collect()

    if len(seed_results) == 0:
        raise RuntimeError("No TabNet seed results found")

    all_results[model_name] = seed_results

    # Restore model object from best-seed checkpoint
    best_seed = int(max(seed_results, key=lambda r: r["auc_roc"])["seed"])
    best_paths = get_seed_ckpt_paths(model_name, best_seed)
    best_meta = load_seed_meta(model_name, best_seed)
    best_zip = Path(best_meta.get("model_zip", best_paths["tabnet_zip"]))

    best_model = TabNetClassifier()
    best_model.load_model(str(best_zip))

    compact_bundle = joblib.load(best_paths["bundle_joblib"])
    infer_bundle = build_deep_infer_bundle_from_compact(compact_bundle, X_test)

    best_models[model_name] = best_model
    best_preprocessors[model_name] = infer_bundle
    print(f"[RESUME] {model_name} best seed selected: {best_seed}")
    print_seed_metrics(model_name, seed_results)

run_tabnet_experiment()


==================== TabNet | seed=123 ====================


[I 2026-04-05 08:57:26,462] A new study created in memory with name: no-name-169778e8-50eb-4d8d-96ce-35639eea1434


epoch 0  | loss: 0.19187 | valid_auc: 0.5076  |  0:00:16s
epoch 10 | loss: 0.15586 | valid_auc: 0.5684  |  0:02:52s
epoch 20 | loss: 0.15491 | valid_auc: 0.58348 |  0:05:28s
epoch 30 | loss: 0.15439 | valid_auc: 0.59469 |  0:08:04s
epoch 40 | loss: 0.15409 | valid_auc: 0.59716 |  0:10:38s

Early stopping occurred at epoch 45 with best_epoch = 33 and best_valid_auc = 0.59998


[I 2026-04-05 09:09:53,709] Trial 0 finished with value: 0.5999808648032001 and parameters: {'n_d': 48, 'n_a': 64, 'n_steps': 4, 'gamma': 1.6320252002240783, 'lambda_sparse': 7.700428898922044e-05, 'lr': 0.0023332121502140425, 'batch_size': 2048, 'virtual_batch_size': 128}. Best is trial 0 with value: 0.5999808648032001.


[TabNet|seed=123] trial=0 auc=0.599981 train_time=745.11s
[Optuna] Trial 000 | value=0.599981 | params={'n_d': 48, 'n_a': 64, 'n_steps': 4, 'gamma': 1.6320252002240783, 'lambda_sparse': 7.700428898922044e-05, 'lr': 0.0023332121502140425, 'batch_size': 2048, 'virtual_batch_size': 128}
epoch 0  | loss: 0.20151 | valid_auc: 0.52673 |  0:00:14s
epoch 10 | loss: 0.15557 | valid_auc: 0.57615 |  0:02:38s
epoch 20 | loss: 0.15547 | valid_auc: 0.58275 |  0:05:03s
epoch 30 | loss: 0.15483 | valid_auc: 0.58601 |  0:07:26s
epoch 40 | loss: 0.15542 | valid_auc: 0.57754 |  0:09:51s

Early stopping occurred at epoch 40 with best_epoch = 28 and best_valid_auc = 0.59263


...

[TabNet|seed=789] trial=2 auc=0.577465 train_time=429.76s
[Optuna] Trial 002 | value=0.577465 | params={'n_d': 48, 'n_a': 56, 'n_steps': 5, 'gamma': 1.6427083497681367, 'lambda_sparse': 0.0004862462723757967, 'lr': 0.0007511728861793214, 'batch_size': 4096, 'virtual_batch_size': 512}
epoch 0  | loss: 1.3355  | valid_auc: 0.50029 |  0:00:13s
epoch 10 | loss: 0.15795 | valid_auc: 0.55556 |  0:02:34s
epoch 20 | loss: 0.15574 | valid_auc: 0.57838 |  0:04:54s
epoch 30 | loss: 0.15531 | valid_auc: 0.57854 |  0:07:04s
epoch 40 | loss: 0.15507 | valid_auc: 0.58361 |  0:09:13s
epoch 50 | loss: 0.15492 | valid_auc: 0.59294 |  0:11:23s

Early stopping occurred at epoch 58 with best_epoch = 46 and best_valid_auc = 0.59524


[I 2026-04-05 13:01:22,760] Trial 3 finished with value: 0.5952422956018768 and parameters: {'n_d': 56, 'n_a': 24, 'n_steps': 7, 'gamma': 1.2346607486981376, 'lambda_sparse': 0.00044790117268557505, 'lr': 0.0015950687062487583, 'batch_size': 4096, 'virtual_batch_size': 256}. Best is trial 0 with value: 0.6104496575146166.


[TabNet|seed=789] trial=3 auc=0.595242 train_time=809.20s
[Optuna] Trial 003 | value=0.595242 | params={'n_d': 56, 'n_a': 24, 'n_steps': 7, 'gamma': 1.2346607486981376, 'lambda_sparse': 0.00044790117268557505, 'lr': 0.0015950687062487583, 'batch_size': 4096, 'virtual_batch_size': 256}
epoch 0  | loss: 0.3511  | valid_auc: 0.50103 |  0:00:09s
epoch 10 | loss: 0.1724  | valid_auc: 0.52351 |  0:01:40s
epoch 20 | loss: 0.16502 | valid_auc: 0.53824 |  0:03:11s
epoch 30 | loss: 0.1619  | valid_auc: 0.54774 |  0:04:43s
epoch 40 | loss: 0.16036 | valid_auc: 0.55526 |  0:06:15s
epoch 50 | loss: 0.15916 | valid_auc: 0.55988 |  0:07:46s
epoch 60 | loss: 0.15822 | valid_auc: 0.56235 |  0:09:17s
epoch 70 | loss: 0.15753 | valid_auc: 0.56863 |  0:10:48s
Stop training because you reached max_epochs = 80 with best_epoch = 79 and best_valid_auc = 0.57093


[I 2026-04-05 13:13:55,977] Trial 4 finished with value: 0.5709323497455868 and parameters: {'n_d': 48, 'n_a': 56, 'n_steps': 4, 'gamma': 1.202394214126921, 'lambda_sparse': 0.004719311882541049, 'lr': 0.0001328072816747972, 'batch_size': 2048, 'virtual_batch_size': 512}. Best is trial 0 with value: 0.6104496575146166.


[TabNet|seed=789] trial=4 auc=0.570932 train_time=751.91s
[Optuna] Trial 004 | value=0.570932 | params={'n_d': 48, 'n_a': 56, 'n_steps': 4, 'gamma': 1.202394214126921, 'lambda_sparse': 0.004719311882541049, 'lr': 0.0001328072816747972, 'batch_size': 2048, 'virtual_batch_size': 512}
[BEST] TabNet seed=789 best_valid_auc=0.610450
[BEST] params={'n_d': 56, 'n_a': 32, 'n_steps': 5, 'gamma': 1.1049875022039695, 'lambda_sparse': 0.0007662583360741423, 'lr': 0.0009752395539501081, 'batch_size': 1024, 'virtual_batch_size': 256}
epoch 0  | loss: 0.3275  | valid_auc: 0.50191 |  0:00:18s
epoch 10 | loss: 0.15621 | valid_auc: 0.5737  |  0:03:27s
epoch 20 | loss: 0.15576 | valid_auc: 0.5794  |  0:06:34s
epoch 30 | loss: 0.15494 | valid_auc: 0.59497 |  0:09:41s
epoch 40 | loss: 0.15466 | valid_auc: 0.60451 |  0:12:47s
epoch 50 | loss: 0.15411 | valid_auc: 0.61045 |  0:15:57s
epoch 60 | loss: 0.15389 | valid_auc: 0.60186 |  0:19:03s

Early stopping occurred at epoch 62 with best_epoch = 50 and best_v

In [ ]:
def run_fttransformer_experiment(n_trials=N_TRIALS, seeds=SEEDS):
    import joblib
    model_name = "FTTransformer"
    seed_results = []

    for seed in seeds:
        paths = get_seed_ckpt_paths(model_name, seed)
        cached_meta = load_seed_meta(model_name, seed)
        if cached_meta is not None and paths["ft_state"].exists() and paths["bundle_joblib"].exists():
            cached_metrics = cached_meta.get("metrics", {})
            if "seed" not in cached_metrics:
                cached_metrics["seed"] = int(seed)
            print(f"[RESUME] {model_name} seed={seed} loaded from checkpoint, skip training")
            seed_results.append(cached_metrics)
            continue

        print(f"\n{'='*20} {model_name} | seed={seed} {'='*20}")
        np.random.seed(seed)
        random.seed(seed)
        torch.manual_seed(seed)

        X_train_seed = X_train
        y_train_seed = y_train
        X_valid_seed = X_valid
        y_valid_seed = y_valid

        # More aggressive subsampling for FT only, to avoid memory/VRAM pressure causing kernel crashes
        train_cap = MAX_TRAIN_ROWS_FT if MAX_TRAIN_ROWS_FT is not None else MAX_TRAIN_ROWS
        valid_cap = MAX_VALID_ROWS_FT if MAX_VALID_ROWS_FT is not None else MAX_VALID_ROWS

        if train_cap is not None and len(X_train_seed) > train_cap:
            idx = X_train_seed.sample(n=train_cap, random_state=seed).index
            X_train_seed = X_train_seed.loc[idx]
            y_train_seed = y_train_seed.loc[idx]

        if valid_cap is not None and len(X_valid_seed) > valid_cap:
            idx = X_valid_seed.sample(n=valid_cap, random_state=seed).index
            X_valid_seed = X_valid_seed.loc[idx]
            y_valid_seed = y_valid_seed.loc[idx]

        bundle = prepare_deep_inputs(X_train_seed, X_valid_seed, X_test)
        ytr_np = y_train_seed.values.astype(int)
        yva_np = y_valid_seed.values.astype(int)
        yte_np = y_test.values.astype(int)

        def objective(trial):
            epoch_high = max(1, int(FT_MAX_EPOCHS))
            epoch_low = 1 if epoch_high < 8 else 8
            patience_high = min(6, epoch_high)
            patience_low = 1 if patience_high < 3 else 3

            params = {
                "dim": trial.suggest_categorical("dim", [8, 16]),
                "depth": trial.suggest_int("depth", 1, 3),
                "heads": trial.suggest_categorical("heads", [4]),
                "attn_dropout": trial.suggest_float("attn_dropout", 0.0, 0.3),
                "ff_dropout": trial.suggest_float("ff_dropout", 0.0, 0.3),
                "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [32, 64]),
                "epochs": trial.suggest_int("epochs", epoch_low, epoch_high),
                "patience": trial.suggest_int("patience", patience_low, patience_high),
            }
            t0 = time.perf_counter()
            # fit_fttransformer_once internally handles move to GPU and returns best AUC
            model, best_auc_valid = fit_fttransformer_once(bundle, ytr_np, yva_np, params, seed)
            t1 = time.perf_counter()

            print(
                f"[FTTransformer|seed={seed}] trial={trial.number} valid_auc={best_auc_valid:.6f} "
                f"train_time={t1-t0:.2f}s"
            )

            # Clean up immediately
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return best_auc_valid

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, callbacks=[optuna_log_callback])
        best_params = study.best_trial.params
        print(f"[BEST] FTTransformer seed={seed} best_valid_auc={study.best_value:.6f}")
        print(f"[BEST] params={best_params}")

        t0 = time.perf_counter()
        final_model, _ = fit_fttransformer_once(bundle, ytr_np, yva_np, best_params, seed)
        y_prob_test = ft_predict_proba(final_model, bundle["Xte_cat"], bundle["Xte_num"])
        t1 = time.perf_counter()

        m = evaluate_predictions(yte_np, y_prob_test)
        m["seed"] = int(seed)
        m["train_seconds"] = float(t1 - t0)

        lat_list = []
        for _ in range(INFER_REPEAT):
            start = time.perf_counter()
            _ = ft_predict_proba(final_model, bundle["Xte_cat"], bundle["Xte_num"])
            end = time.perf_counter()
            lat_list.append((end - start) / len(bundle["Xte_num"]) * 1000)
        m["infer_ms_per_sample"] = float(np.mean(lat_list))
        m["infer_ms_per_sample_std"] = float(np.std(lat_list))

        torch.save(final_model.state_dict(), paths["ft_state"])
        m["model_size_mb"] = float(os.path.getsize(paths["ft_state"]) / (1024 * 1024))
        m["best_valid_auc"] = float(study.best_value)
        m["best_params"] = best_params
        seed_results.append(m)

        compact_bundle = compact_deep_bundle(bundle)
        joblib.dump(compact_bundle, paths["bundle_joblib"])
        save_seed_meta(
            model_name,
            seed,
            {
                "model_name": model_name,
                "seed": int(seed),
                "format": "ft_transformer",
                "metrics": m,
                "best_params": best_params,
                "state_path": str(paths["ft_state"]),
            },
        )
        print(f"[CKPT] Saved {model_name} seed={seed} -> {paths['dir']}")

        del final_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if len(seed_results) == 0:
        raise RuntimeError("No FTTransformer seed results found")

    all_results[model_name] = seed_results

    # Reload best seed for the dictionary
    best_seed = int(max(seed_results, key=lambda r: r["auc_roc"])["seed"])
    best_paths = get_seed_ckpt_paths(model_name, best_seed)
    best_meta = load_seed_meta(model_name, best_seed)
    best_p = best_meta["best_params"]

    compact_bundle = joblib.load(best_paths["bundle_joblib"])
    best_model = FTTransformer(
        categories=tuple(compact_bundle["cat_dims"]),
        num_continuous=compact_bundle["n_num"],
        dim=best_p["dim"],
        dim_out=1,
        depth=best_p["depth"],
        heads=best_p["heads"],
        attn_dropout=best_p["attn_dropout"],
        ff_dropout=best_p["ff_dropout"],
    ).to(DEVICE)
    best_model.load_state_dict(torch.load(best_paths["ft_state"], map_location=DEVICE))

    best_models[model_name] = best_model
    best_preprocessors[model_name] = build_deep_infer_bundle_from_compact(compact_bundle, X_test)
    print(f"[RESUME] {model_name} best seed selected: {best_seed}")
    print_seed_metrics(model_name, seed_results)

run_fttransformer_experiment()


==================== FTTransformer | seed=123 ====================


[I 2026-04-05 14:19:06,085] A new study created in memory with name: no-name-a9d09517-6ca0-4c56-a430-2588863cc2c2


[FTTransformer|seed=123] epoch=001 loss=0.268460 valid_auc=0.547257
[FTTransformer|seed=123] epoch=002 loss=0.223667 valid_auc=0.545115
[FTTransformer|seed=123] epoch=003 loss=0.196577 valid_auc=0.466172
[FTTransformer|seed=123] epoch=004 loss=0.179464 valid_auc=0.548071
[FTTransformer|seed=123] epoch=005 loss=0.169153 valid_auc=0.557286
[FTTransformer|seed=123] epoch=006 loss=0.163239 valid_auc=0.500995
[FTTransformer|seed=123] epoch=007 loss=0.159984 valid_auc=0.558710
[FTTransformer|seed=123] epoch=008 loss=0.157897 valid_auc=0.566581
[FTTransformer|seed=123] epoch=009 loss=0.156095 valid_auc=0.566664
[FTTransformer|seed=123] epoch=010 loss=0.153801 valid_auc=0.571586
[FTTransformer|seed=123] epoch=011 loss=0.152998 valid_auc=0.579268
[FTTransformer|seed=123] epoch=012 loss=0.151696 valid_auc=0.570925
[FTTransformer|seed=123] epoch=013 loss=0.151061 valid_auc=0.583793
[FTTransformer|seed=123] epoch=014 loss=0.150122 valid_auc=0.580291
[FTTransformer|seed=123] epoch=015 loss=0.149756

[I 2026-04-05 14:24:29,075] Trial 0 finished with value: 0.5837925513853497 and parameters: {'dim': 8, 'depth': 3, 'heads': 4, 'attn_dropout': 0.05312059183937995, 'ff_dropout': 0.11792344503510276, 'lr': 0.00020867005316315684, 'weight_decay': 5.7985047923593995e-06, 'batch_size': 64, 'epochs': 19, 'patience': 3}. Best is trial 0 with value: 0.5837925513853497.


[Optuna] Trial 000 | value=0.583793 | params={'dim': 8, 'depth': 3, 'heads': 4, 'attn_dropout': 0.05312059183937995, 'ff_dropout': 0.11792344503510276, 'lr': 0.00020867005316315684, 'weight_decay': 5.7985047923593995e-06, 'batch_size': 64, 'epochs': 19, 'patience': 3}
[FTTransformer|seed=123] epoch=001 loss=0.202295 valid_auc=0.494063
[FTTransformer|seed=123] epoch=002 loss=0.157010 valid_auc=0.569729
[FTTransformer|seed=123] epoch=003 loss=0.156095 valid_auc=0.577155
[FTTransformer|seed=123] epoch=004 loss=0.154086 valid_auc=0.585396
[FTTransformer|seed=123] epoch=005 loss=0.152619 valid_auc=0.581387
[FTTransformer|seed=123] epoch=006 loss=0.151777 valid_auc=0.585058
[FTTransformer|seed=123] epoch=007 loss=0.151163 valid_auc=0.584533
[FTTransformer|seed=123] epoch=008 loss=0.150970 valid_auc=0.579238
[FTTransformer|seed=123] Early stopping at epoch 8
[FTTransformer|seed=123] trial=1 valid_auc=0.585396 train_time=112.11s


...

[Optuna] Trial 002 | value=0.599940 | params={'dim': 8, 'depth': 2, 'heads': 4, 'attn_dropout': 0.19181446363684787, 'ff_dropout': 0.03854710512517216, 'lr': 0.0007973526278132735, 'weight_decay': 0.000525122945401058, 'batch_size': 32, 'epochs': 16, 'patience': 4}
[FTTransformer|seed=789] epoch=001 loss=0.221113 valid_auc=0.559138
[FTTransformer|seed=789] epoch=002 loss=0.166056 valid_auc=0.484843
[FTTransformer|seed=789] epoch=003 loss=0.164868 valid_auc=0.595613
[FTTransformer|seed=789] epoch=004 loss=0.164275 valid_auc=0.600058
[FTTransformer|seed=789] epoch=005 loss=0.163685 valid_auc=0.609956
[FTTransformer|seed=789] epoch=006 loss=0.162027 valid_auc=0.608752
[FTTransformer|seed=789] epoch=007 loss=0.161541 valid_auc=0.604843
[FTTransformer|seed=789] epoch=008 loss=0.160484 valid_auc=0.610209
[FTTransformer|seed=789] epoch=009 loss=0.159961 valid_auc=0.606987
[FTTransformer|seed=789] epoch=010 loss=0.159668 valid_auc=0.591356
[FTTransformer|seed=789] epoch=011 loss=0.159003 valid

[I 2026-04-05 15:18:32,647] Trial 3 finished with value: 0.6148772269188704 and parameters: {'dim': 16, 'depth': 3, 'heads': 4, 'attn_dropout': 0.12996486059101658, 'ff_dropout': 0.0826472990852878, 'lr': 0.0005228726043341119, 'weight_decay': 0.0006486407934084053, 'batch_size': 32, 'epochs': 12, 'patience': 4}. Best is trial 3 with value: 0.6148772269188704.


[Optuna] Trial 003 | value=0.614877 | params={'dim': 16, 'depth': 3, 'heads': 4, 'attn_dropout': 0.12996486059101658, 'ff_dropout': 0.0826472990852878, 'lr': 0.0005228726043341119, 'weight_decay': 0.0006486407934084053, 'batch_size': 32, 'epochs': 12, 'patience': 4}
[FTTransformer|seed=789] epoch=001 loss=0.206426 valid_auc=0.537307
[FTTransformer|seed=789] epoch=002 loss=0.165365 valid_auc=0.574508
[FTTransformer|seed=789] epoch=003 loss=0.164666 valid_auc=0.588282
[FTTransformer|seed=789] epoch=004 loss=0.163367 valid_auc=0.578208
[FTTransformer|seed=789] epoch=005 loss=0.162181 valid_auc=0.551476
[FTTransformer|seed=789] epoch=006 loss=0.161557 valid_auc=0.593466
[FTTransformer|seed=789] epoch=007 loss=0.160701 valid_auc=0.594321
[FTTransformer|seed=789] epoch=008 loss=0.159776 valid_auc=0.598372
[FTTransformer|seed=789] epoch=009 loss=0.158759 valid_auc=0.612651
[FTTransformer|seed=789] trial=4 valid_auc=0.612651 train_time=246.68s


[I 2026-04-05 15:22:39,618] Trial 4 finished with value: 0.6126513753705547 and parameters: {'dim': 16, 'depth': 2, 'heads': 4, 'attn_dropout': 0.10774598802271507, 'ff_dropout': 0.23084781912712685, 'lr': 0.00048096026175157414, 'weight_decay': 0.0076085734920716295, 'batch_size': 32, 'epochs': 9, 'patience': 5}. Best is trial 3 with value: 0.6148772269188704.


[Optuna] Trial 004 | value=0.612651 | params={'dim': 16, 'depth': 2, 'heads': 4, 'attn_dropout': 0.10774598802271507, 'ff_dropout': 0.23084781912712685, 'lr': 0.00048096026175157414, 'weight_decay': 0.0076085734920716295, 'batch_size': 32, 'epochs': 9, 'patience': 5}
[BEST] FTTransformer seed=789 best_valid_auc=0.614877
[BEST] params={'dim': 16, 'depth': 3, 'heads': 4, 'attn_dropout': 0.12996486059101658, 'ff_dropout': 0.0826472990852878, 'lr': 0.0005228726043341119, 'weight_decay': 0.0006486407934084053, 'batch_size': 32, 'epochs': 12, 'patience': 4}
[FTTransformer|seed=789] epoch=001 loss=0.221113 valid_auc=0.559138
[FTTransformer|seed=789] epoch=002 loss=0.166056 valid_auc=0.484843
[FTTransformer|seed=789] epoch=003 loss=0.164868 valid_auc=0.595613
[FTTransformer|seed=789] epoch=004 loss=0.164275 valid_auc=0.600058
[FTTransformer|seed=789] epoch=005 loss=0.163685 valid_auc=0.609956
[FTTransformer|seed=789] epoch=006 loss=0.162027 valid_auc=0.608752
[FTTransformer|seed=789] epoch=007

In [ ]:
# ===== 3.6 Deep Model Utility: FT-Transformer Memory Cleanup =====
# Clean previously trained models from memory to free GPU space for FT-Transformer
print("[INFO] Clearing best_models and best_preprocessors to free GPU memory for FT-Transformer...")

# Explicitly delete model objects that might hold GPU memory
for model_name, model_obj in best_models.items():
    if hasattr(model_obj, 'to') and callable(getattr(model_obj, 'to')):
        # For PyTorch models, move to CPU or detach
        try:
            model_obj.to('cpu')
            del model_obj # Explicitly delete
        except Exception as e:
            print(f"[WARN] Could not move {model_name} model to CPU or delete: {e}")
    elif 'XGBoost' in model_name or 'LightGBM' in model_name or 'CatBoost' in model_name:
        # For tree models, explicit deletion might help
        try:
            del model_obj
        except Exception as e:
            print(f"[WARN] Could not delete {model_name} model object: {e}")

# Clear global dictionaries
best_models.clear()
best_preprocessors.clear()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"[INFO] CUDA memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.2f} MB allocated")
else:
    print("[INFO] GPU not available or PyTorch not using CUDA.")

print("[INFO] Memory cleanup complete. Proceeding with FT-Transformer experiment.")



[INFO] Clearing best_models and best_preprocessors to free GPU memory for FT-Transformer...
[INFO] CUDA memory after cleanup: 18.89 MB allocated
[INFO] Memory cleanup complete. Proceeding with FT-Transformer experiment.


### Fixing post-cleanup prediction/analysis issues: reload all best models



To fix the issue where cleaning models before FT-Transformer training breaks later prediction and analysis, reload each model's best checkpoint from disk after all model training (including FT-Transformer) completes. This ensures all models are available in the `best_models` and `best_preprocessors` dictionaries for summary and prediction steps.

In [ ]:
# ===== Reload all best models and preprocessors =====
import joblib

print("[INFO] Reloading all best models and preprocessors from checkpoints...")

# Clear existing in-memory objects to ensure fresh reload and avoid duplicates
best_models.clear()
best_preprocessors.clear()

for model_name, seed_results in all_results.items():
    if not seed_results:
        print(f"[WARN] No results found for {model_name}. Skipping reload.")
        continue

    # Find the best seed based on AUC-ROC from the stored results
    best_seed_result = max(seed_results, key=lambda r: r["auc_roc"])
    best_seed = int(best_seed_result["seed"])
    best_meta = best_seed_result.get("best_params", {})

    paths = get_seed_ckpt_paths(model_name, best_seed)

    # Handle different model types for loading
    if model_name in ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]:
        try:
            best_pack = joblib.load(paths["model_joblib"])
            best_models[model_name] = best_pack["model"]
            best_preprocessors[model_name] = best_pack["preprocessor"]
            print(f"[INFO] Reloaded {model_name} (seed={best_seed}) from {paths['model_joblib']}")
        except Exception as e:
            print(f"[ERROR] Failed to reload {model_name} from {paths['model_joblib']}: {e}")
    elif model_name == "TabNet":
        try:
            best_model = TabNetClassifier()
            # Use the exact path from meta if available, otherwise default
            meta_path = load_seed_meta(model_name, best_seed).get("model_zip", paths["tabnet_zip"])
            best_model.load_model(str(meta_path))
            best_models[model_name] = best_model

            compact_bundle = joblib.load(paths["bundle_joblib"])
            best_preprocessors[model_name] = build_deep_infer_bundle_from_compact(compact_bundle, X_test)
            print(f"[INFO] Reloaded {model_name} (seed={best_seed}) from {meta_path}")
        except Exception as e:
            print(f"[ERROR] Failed to reload {model_name} from {paths['tabnet_zip']}: {e}")
    elif model_name == "FTTransformer":
        try:
            best_params = load_seed_meta(model_name, best_seed)["best_params"]
            compact_bundle = joblib.load(paths["bundle_joblib"])

            best_model = FTTransformer(
                categories=tuple(compact_bundle["cat_dims"]),
                num_continuous=compact_bundle["n_num"],
                dim=best_params["dim"],
                dim_out=1,
                depth=best_params["depth"],
                heads=best_params["heads"],
                attn_dropout=best_params["attn_dropout"],
                ff_dropout=best_params["ff_dropout"],
            ).to(DEVICE)
            state_dict = torch.load(paths["ft_state"], map_location=DEVICE)
            best_model.load_state_dict(state_dict)

            best_models[model_name] = best_model
            best_preprocessors[model_name] = build_deep_infer_bundle_from_compact(compact_bundle, X_test)
            print(f"[INFO] Reloaded {model_name} (seed={best_seed}) from {paths['ft_state']}")
        except Exception as e:
            print(f"[ERROR] Failed to reload {model_name} from {paths['ft_state']}: {e}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("[INFO] All best models and preprocessors reloaded. You can now proceed with prediction and analysis.")

[INFO] Reloading all best models and preprocessors from checkpoints...
[INFO] Reloaded LogisticRegression (seed=456) from artifacts/checkpoints/LogisticRegression_seed456/model.joblib
[INFO] Reloaded RandomForest (seed=789) from artifacts/checkpoints/RandomForest_seed789/model.joblib
[INFO] Reloaded XGBoost (seed=456) from artifacts/checkpoints/XGBoost_seed456/model.joblib
[INFO] Reloaded LightGBM (seed=789) from artifacts/checkpoints/LightGBM_seed789/model.joblib
[INFO] Reloaded CatBoost (seed=456) from artifacts/checkpoints/CatBoost_seed456/model.joblib
[INFO] Reloaded TabNet (seed=456) from artifacts/checkpoints/TabNet_seed456/tabnet_model.zip
[INFO] Reloaded FTTransformer (seed=456) from artifacts/checkpoints/FTTransformer_seed456/ft_state.pt
[INFO] All best models and preprocessors reloaded. You can now proceed with prediction and analysis.


In [56]:
!nvidia-smi

Sun Apr  5 14:00:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             29W /   70W |   14853MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4) Model Prediction and Comprehensive Analysis



Outputs from this stage:

- Overall test-set result table with mean 卤 standard deviation for each model

- Inference cost and model size comparison

- Stability analysis (cross-seed variance)

- Feature-importance consistency comparison

- Practice-oriented conclusion recommendations

In [ ]:
# ===== 4.1 Summary Results Table =====
rows = []
for model_name, seed_res in all_results.items():
    if len(seed_res) == 0:
        continue
    row = {"model": model_name}
    for m in ["accuracy", "auc_roc", "f1", "train_seconds", "infer_ms_per_sample", "model_size_mb", "best_valid_auc"]:
        vals = [r[m] for r in seed_res]
        row[f"{m}_mean"] = float(np.mean(vals))
        row[f"{m}_std"] = float(np.std(vals))
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values("auc_roc_mean", ascending=False).reset_index(drop=True)
display(summary_df)

summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"[INFO] Saved summary to {SUMMARY_PATH}")

,model,accuracy_mean,accuracy_std,auc_roc_mean,auc_roc_std,f1_mean,f1_std,train_seconds_mean,train_seconds_std,infer_ms_per_sample_mean,infer_ms_per_sample_std,model_size_mb_mean,model_size_mb_std,best_valid_auc_mean,best_valid_auc_std
0,CatBoost,0.96,0.00e+00,0.64,1.08e-03,1.54e-04,2.17e-04,114.07,26.51,1.45e-02,1.73e-03,12.22,9.39,0.64,3.11e-04
1,XGBoost,0.96,0.00e+00,0.63,3.35e-04,0.00e+00,0.00e+00,9.12,2.12,8.37e-03,8.04e-04,4.28,4.62,0.63,2.63e-03
2,LightGBM,0.96,3.96e-06,0.63,1.59e-03,0.00e+00,0.00e+00,22.71,11.88,1.76e-02,7.87e-03,2.51,1.80,0.63,1.56e-03
3,LogisticRegression,0.62,6.99e-04,0.63,1.92e-03,9.60e-02,6.52e-04,816.12,859.86,6.75e-03,1.22e-04,0.01,0.00,0.62,2.09e-04
4,RandomForest,0.66,5.31e-02,0.62,1.50e-03,9.68e-02,1.75e-03,177.39,46.69,1.31e-01,3.25e-02,31.09,26.74,0.62,6.05e-04
5,TabNet,0.96,0.00e+00,0.61,1.11e-02,0.00e+00,0.00e+00,783.77,330.55,1.65e-02,4.05e-03,0.62,0.25,0.61,1.00e-02
6,FTTransformer,0.96,0.00e+00,0.59,1.71e-03,0.00e+00,0.00e+00,307.33,151.84,4.03e-02,1.77e-02,0.09,0.04,0.61,1.55e-02


[INFO] Saved summary to artifacts/model_summary.csv


In [ ]:
# ===== 4.2 Export Test-Set Predictions (Probabilities) =====
test_pred_df = pd.DataFrame(index=X_test.index)
test_pred_df[TARGET_COL] = y_test.values

for model_name, model in best_models.items():
    print(f"[PREDICT] {model_name}")
    if model_name in ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]:
        prep = best_preprocessors[model_name]
        proba = model.predict_proba(prep.transform(X_test))[:, 1]
    elif model_name == "CatBoost":
        Xt = X_test.copy()
        for c in cat_cols:
            Xt[c] = Xt[c].astype("string").fillna("MISSING")
        proba = model.predict_proba(Xt)[:, 1]
    elif model_name in ["TabNet"]:
        bundle = best_preprocessors[model_name]
        Xte_tab = np.hstack([bundle["Xte_num"], bundle["Xte_cat"]]).astype(np.float32)
        proba = model.predict_proba(Xte_tab)[:, 1]
    elif model_name in ["FTTransformer"]:
        bundle = best_preprocessors[model_name]
        proba = ft_predict_proba(model, bundle["Xte_cat"], bundle["Xte_num"])
    else:
        continue
    test_pred_df[f"proba_{model_name}"] = proba

test_pred_df.to_csv(PRED_PATH, index=True)
print(f"[INFO] Saved test predictions to {PRED_PATH}")
display(test_pred_df.head())

[PREDICT] LogisticRegression
[PREDICT] RandomForest
[PREDICT] XGBoost
[PREDICT] LightGBM
[PREDICT] CatBoost
[PREDICT] TabNet
[PREDICT] FTTransformer
[INFO] Saved test predictions to artifacts/test_predictions_all_models.csv


,target,proba_LogisticRegression,proba_RandomForest,proba_XGBoost,proba_LightGBM,proba_CatBoost,proba_TabNet,proba_FTTransformer
235465,0,0.26,0.44,0.02,0.02,0.02,0.02,0.03
292820,0,0.49,0.50,0.05,0.04,0.04,0.04,0.03
434381,0,0.38,0.53,0.03,0.03,0.03,0.05,0.05
472579,0,0.44,0.48,0.03,0.03,0.03,0.02,0.03
238219,0,0.62,0.56,0.06,0.05,0.05,0.08,0.04


In [ ]:
# ===== 4.3 Preprocessing Cost and Robustness Analysis =====
preprocess_notes = pd.DataFrame([
    {"model": "LogisticRegression", "missing_handling": "median/mode imputation", "categorical": "one-hot", "effort_level": "high"},
    {"model": "RandomForest", "missing_handling": "median/mode imputation", "categorical": "ordinal encoding", "effort_level": "medium"},
    {"model": "XGBoost", "missing_handling": "native missing + imputation in this pipeline", "categorical": "ordinal encoding", "effort_level": "medium"},
    {"model": "LightGBM", "missing_handling": "native missing + imputation in this pipeline", "categorical": "ordinal encoding", "effort_level": "medium"},
    {"model": "CatBoost", "missing_handling": "native support", "categorical": "native categorical", "effort_level": "low"},
    {"model": "TabNet", "missing_handling": "numeric imputation + missing category token", "categorical": "learned embeddings", "effort_level": "high"},
    {"model": "FTTransformer", "missing_handling": "numeric imputation + missing category token", "categorical": "learned embeddings", "effort_level": "high"},
])

display(preprocess_notes)



if not summary_df.empty:
    stability = summary_df[["model", "auc_roc_std", "f1_std", "accuracy_std"]].sort_values("auc_roc_std")
    print("[INFO] Lower std => more stable across seeds")
    display(stability)

,model,missing_handling,categorical,effort_level
0,LogisticRegression,median/mode imputation,one-hot,high
1,RandomForest,median/mode imputation,ordinal encoding,medium
2,XGBoost,native missing + imputation in this pipeline,ordinal encoding,medium
3,LightGBM,native missing + imputation in this pipeline,ordinal encoding,medium
4,CatBoost,native support,native categorical,low
5,TabNet,numeric imputation + missing category token,learned embeddings,high
6,FTTransformer,numeric imputation + missing category token,learned embeddings,high


[INFO] Lower std => more stable across seeds


,model,auc_roc_std,f1_std,accuracy_std
1,XGBoost,3.35e-04,0.00e+00,0.00e+00
0,CatBoost,1.08e-03,2.17e-04,0.00e+00
4,RandomForest,1.50e-03,1.75e-03,5.31e-02
2,LightGBM,1.59e-03,0.00e+00,3.96e-06
6,FTTransformer,1.71e-03,0.00e+00,0.00e+00
3,LogisticRegression,1.92e-03,6.52e-04,6.99e-04
5,TabNet,1.11e-02,0.00e+00,0.00e+00


In [ ]:
# ===== 4.4 Feature Importance Comparison (Unified Permutation Importance on Validation Subsample) =====
def predict_proba_on_raw(model_name, Xraw):
    model = best_models[model_name]
    if model_name in ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]:
        prep = best_preprocessors[model_name]
        return model.predict_proba(prep.transform(Xraw))[:, 1]

    if model_name == "CatBoost":
        Xt = Xraw.copy()
        for c in cat_cols:
            Xt[c] = Xt[c].astype("string").fillna("MISSING")
        return model.predict_proba(Xt)[:, 1]

    if model_name == "TabNet":
        bundle = best_preprocessors[model_name]
        b = prepare_deep_inputs(X_train, Xraw, Xraw)
        X_tab = np.hstack([b["Xva_num"], b["Xva_cat"]]).astype(np.float32)
        return model.predict_proba(X_tab)[:, 1]
    if model_name == "FTTransformer":
        b = prepare_deep_inputs(X_train, Xraw, Xraw)
        return ft_predict_proba(model, b["Xva_cat"], b["Xva_num"])
    raise ValueError(model_name)



def permutation_importance_auc(model_name, Xraw, ytrue, max_features=None, seed=42):
    rng = np.random.default_rng(seed)
    base_pred = predict_proba_on_raw(model_name, Xraw)
    base_auc = roc_auc_score(ytrue, base_pred)
    cols = list(Xraw.columns)
    if max_features is not None and len(cols) > max_features:
        cols = cols[:max_features]

    imps = []

    for c in cols:
        Xp = Xraw.copy()
        Xp[c] = rng.permutation(Xp[c].values)
        auc_p = roc_auc_score(ytrue, predict_proba_on_raw(model_name, Xp))
        imps.append((c, base_auc - auc_p))
    imp_df = pd.DataFrame(imps, columns=["feature", "importance_drop_auc"]).sort_values(
        "importance_drop_auc", ascending=False
    )

    return base_auc, imp_df

X_val_for_imp = X_valid.sample(min(len(X_valid), 6000), random_state=SEEDS[0])
y_val_for_imp = y_valid.loc[X_val_for_imp.index]

importance_tables = {}

for model_name in all_results.keys():
    print(f"\n[IMP] {model_name}")
    base_auc, imp_df = permutation_importance_auc(model_name, X_val_for_imp, y_val_for_imp, max_features=None, seed=SEEDS[0])
    importance_tables[model_name] = imp_df
    print(f"[IMP] base_auc={base_auc:.6f}")
    display(imp_df.head(15))


[IMP] LogisticRegression
[IMP] base_auc=0.638093


,feature,importance_drop_auc
5,ps_ind_05_cat,2.64e-02
19,ps_reg_01,1.10e-02
35,ps_car_13,1.09e-02
32,ps_car_11_cat,8.19e-03
22,ps_car_01_cat,7.15e-03
8,ps_ind_08_bin,7.11e-03
34,ps_car_12,6.68e-03
17,ps_ind_17_bin,5.42e-03
25,ps_car_04_cat,5.37e-03
28,ps_car_07_cat,4.52e-03



[IMP] RandomForest
[IMP] base_auc=0.635232


,feature,importance_drop_auc
5,ps_ind_05_cat,1.26e-02
21,ps_reg_03,8.96e-03
20,ps_reg_02,8.35e-03
6,ps_ind_06_bin,7.71e-03
35,ps_car_13,7.29e-03
22,ps_car_01_cat,6.99e-03
19,ps_reg_01,6.56e-03
17,ps_ind_17_bin,6.13e-03
25,ps_car_04_cat,4.16e-03
3,ps_ind_03,3.92e-03



[IMP] XGBoost
[IMP] base_auc=0.649442


...

,feature,importance_drop_auc
5,ps_ind_05_cat,1.96e-02
17,ps_ind_17_bin,9.47e-03
21,ps_reg_03,9.33e-03
3,ps_ind_03,8.79e-03
15,ps_ind_15,5.71e-03
35,ps_car_13,5.58e-03
6,ps_ind_06_bin,4.96e-03
20,ps_reg_02,4.77e-03
28,ps_car_07_cat,4.39e-03
24,ps_car_03_cat,3.68e-03



[IMP] TabNet
[IMP] base_auc=0.633347


,feature,importance_drop_auc
5,ps_ind_05_cat,2.46e-02
35,ps_car_13,1.82e-02
21,ps_reg_03,1.54e-02
19,ps_reg_01,1.31e-02
28,ps_car_07_cat,8.76e-03
4,ps_ind_04_cat,6.28e-03
30,ps_car_09_cat,4.97e-03
8,ps_ind_08_bin,4.79e-03
20,ps_reg_02,4.64e-03
9,ps_ind_09_bin,4.48e-03



[IMP] FTTransformer
[IMP] base_auc=0.579587


,feature,importance_drop_auc
28,ps_car_07_cat,9.94e-03
17,ps_ind_17_bin,8.86e-03
5,ps_ind_05_cat,7.92e-03
25,ps_car_04_cat,5.37e-03
27,ps_car_06_cat,4.55e-03
6,ps_ind_06_bin,4.39e-03
24,ps_car_03_cat,3.21e-03
2,ps_ind_02_cat,2.88e-03
32,ps_car_11_cat,1.17e-03
49,ps_calc_12,8.08e-04


In [ ]:
# ===== 4.5 Auto-Generate Conclusion Draft (Report-Ready) =====
if summary_df.empty:
    print("[WARN] summary_df is empty. Run training cells first.")
else:
    best_row = summary_df.iloc[0]
    fastest_row = summary_df.sort_values("infer_ms_per_sample_mean", ascending=True).iloc[0]
    smallest_row = summary_df.sort_values("model_size_mb_mean", ascending=True).iloc[0]
    most_stable_row = summary_df.sort_values("auc_roc_std", ascending=True).iloc[0]

    print("===== Practical Verdict Draft =====")
    print(f"1) Overall best predictive AUC: {best_row['model']} | AUC={best_row['auc_roc_mean']:.5f}卤{best_row['auc_roc_std']:.5f}")
    print(f"2) Fastest inference: {fastest_row['model']} | {fastest_row['infer_ms_per_sample_mean']:.6f} ms/sample")
    print(f"3) Smallest model size: {smallest_row['model']} | {smallest_row['model_size_mb_mean']:.3f} MB")
    print(f"4) Most stable across seeds: {most_stable_row['model']} | AUC std={most_stable_row['auc_roc_std']:.5f}")

===== Practical Verdict Draft =====
1) Overall best predictive AUC: CatBoost | AUC=0.63666卤0.00108
2) Fastest inference: LogisticRegression | 0.006752 ms/sample
3) Smallest model size: LogisticRegression | 0.011 MB
4) Most stable across seeds: XGBoost | AUC std=0.00033


## 5) Run Recommendations



- Production mode is enabled: `FAST_MODE=False`, suitable for the final report.

- GPU acceleration is enabled where supported (XGBoost, LightGBM, CatBoost, TabNet, FT-Transformer).

- Logistic Regression and Random Forest use sklearn implementations and remain CPU-trained by default.

- It is recommended to run up to Stage 2 first to confirm data and splits are correct.

- All key outputs are written to `artifacts/` using `` file names:

  - `fixed_split_indices.json`

  - `model_summary.csv`

  - `test_predictions_all_models.csv`

- CatBoost training logs are stored in `catboost_info/` to avoid conflicts with other runs.



### Checkpoint Resume Notes (New)



- Training checkpoints are automatically saved to `artifacts/checkpoints/`.

- Checkpoint granularity is model + seed: if interrupted, rerunning the training cell skips completed seeds and continues the remaining ones.

- If all seeds for a model are complete, rerun will directly load the best-seed checkpoint and proceed to summary/prediction.

- To force full retraining of a model, delete its checkpoint subfolder (for example `artifacts/checkpoints/TabNet_seed123/`).